## AND-103 Task 6: ML Pipeline

Predicts inspection outcomes from `data/feature_matrix.csv` using scikit-learn pipelines.

In [1]:
import pandas as pd
import numpy as np

## Step 1 — Load Data and Establish Baseline

Before building any model, establish the **naive baseline**: the accuracy achieved by
always predicting the most common `InspectionOutcome` class, using no features at all.

This is the floor the trained model must beat to provide value.

In [2]:
fm = pd.read_csv('../data/feature_matrix.csv', parse_dates=['InspectionDate'])
print(f'Rows: {len(fm):,}   Columns: {fm.shape[1]}')
print(f'Date range: {fm["InspectionDate"].min().date()} \u2192 {fm["InspectionDate"].max().date()}')

Rows: 143,181   Columns: 96
Date range: 2011-01-04 → 2017-01-09


In [3]:
outcome_counts   = fm['outcome_class'].value_counts()
most_common      = outcome_counts.index[0]
most_common_freq = outcome_counts.iloc[0]
baseline_acc     = most_common_freq / len(fm)

print('Outcome class distribution:')
print(outcome_counts.to_string())
print()
print(f'Most common class : "{most_common}"')
print(f'Frequency         : {most_common_freq:,} of {len(fm):,} ({baseline_acc*100:.1f}%)')
print(f'Baseline accuracy : {baseline_acc*100:.1f}%')
print()
print('Any trained model must exceed this score on the test set to provide value.')

Outcome class distribution:
outcome_class
Follow up              54605
Passed                 26064
DC Follow up           22302
All Orders Resolved    19555
Complete                7506
Shutdown                6110
Other                   2201
Follow up Major         1117
Follow up Sub Major     1002
Follow Up Initial        877
Unable to Inspect        689
Fail Initial             602
Passed Major             551

Most common class : "Follow up"
Frequency         : 54,605 of 143,181 (38.1%)
Baseline accuracy : 38.1%

Any trained model must exceed this score on the test set to provide value.


## Step 2 — Train / Test Split

### Why a random split causes data leakage here

Each row in the feature matrix represents one inspection event. The prior-history
features for that row — `prior_inspection_count`, `days_since_last_inspection`,
`prior_oc_*`, etc. — were built from inspections that occurred *before* that event.

A **random split** would place, say, a 2012 inspection in the test set and a 2016
inspection for the same elevator in the training set. The 2016 row's prior-history
features already aggregate the 2012 inspection's outcome. The model would be trained
on features that encode information from the test set — it has effectively seen the
answer before being asked the question. Performance on such a test set is optimistic
and does not reflect how the model would behave on genuinely unseen future inspections.

A **time-based split** preserves causal order: every training row precedes every test
row in time. The model is evaluated exactly as it would be used in production —
trained on historical inspections, predicting outcomes for later ones.

In [4]:
fm_sorted = fm.sort_values('InspectionDate').reset_index(drop=True)

TRAIN_FRAC = 0.80
split_idx  = int(len(fm_sorted) * TRAIN_FRAC)
cutoff_date = fm_sorted.iloc[split_idx]['InspectionDate']

train_df = fm_sorted.iloc[:split_idx].copy()
test_df  = fm_sorted.iloc[split_idx:].copy()

print(f'Cutoff date : {cutoff_date.date()}')
print(f'Train rows  : {len(train_df):,}  '
      f'({train_df["InspectionDate"].min().date()} – {train_df["InspectionDate"].max().date()})')
print(f'Test rows   : {len(test_df):,}  '
      f'({test_df["InspectionDate"].min().date()} – {test_df["InspectionDate"].max().date()})')
print(f'Train share : {len(train_df)/len(fm_sorted)*100:.1f}%')

Cutoff date : 2015-12-16
Train rows  : 114,544  (2011-01-04 – 2015-12-16)
Test rows   : 28,637  (2015-12-16 – 2017-01-09)
Train share : 80.0%


## Step 3 — Prepare Feature Matrices

Separate identifiers and target from model inputs.
The 58 dummy columns produced by `pd.get_dummies` are stored as `bool` in this pandas
version; they are cast to `int8` so scikit-learn transformers treat them uniformly
as numeric. The only remaining NaN column is `days_since_last_inspection` — handled
inside each pipeline by a `SimpleImputer`.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from functools import partial

IDENTIFIER_COLS = ['ElevatingDevicesNumber', 'InspectionDate']
TARGET_COL      = 'outcome_class'

def make_X(df):
    X = df.drop(columns=IDENTIFIER_COLS + [TARGET_COL])
    bool_cols = X.select_dtypes(bool).columns
    return X.assign(**{c: X[c].astype('int8') for c in bool_cols})

X_train, y_train = make_X(train_df), train_df[TARGET_COL]
X_test,  y_test  = make_X(test_df),  test_df[TARGET_COL]

# Baseline on the test set (always predict the training majority class)
baseline_test = (y_test == most_common).mean()

print(f'Feature columns : {X_train.shape[1]}')
print(f'Train rows      : {X_train.shape[0]:,}')
print(f'Test rows       : {X_test.shape[0]:,}')
print(f'NaN in X_train  : {X_train.isna().sum().sum():,}  (days_since_last_inspection only)')
print(f'Baseline (full) : {baseline_acc*100:.1f}%  (Step 1)')
print(f'Baseline (test) : {baseline_test*100:.1f}%  (predict "{most_common}" on test set)')

Feature columns : 93
Train rows      : 114,544
Test rows       : 28,637
NaN in X_train  : 40,899  (days_since_last_inspection only)
Baseline (full) : 38.1%  (Step 1)
Baseline (test) : 29.0%  (predict "Follow up" on test set)


## Step 4 — Model A: Logistic Regression

### Why Logistic Regression suits this problem

Logistic Regression (multi-class via softmax / 'lbfgs') models the log-odds of each
outcome class as a weighted linear combination of features.

It is a reasonable first model here because:
- **High dimensionality is manageable**: the 93 features include many sparse dummy columns
  (city, equipment type, prior outcome dummies); L2 regularisation shrinks their weights
  toward zero automatically, preventing over-fitting to infrequent categories.
- **Prior counts are nearly linear signals**: the number of prior 'Follow up' inspections
  is probably the strongest single predictor of a future 'Follow up' — a linear
  relationship that Logistic Regression models directly.
- **Fast to train**: completes in seconds on 114 k rows, making it a cheap reference point
  against which more expensive models can be measured.

Its main limitation on this dataset is the assumption of *linearity*: inspection risk
is likely threshold-driven (e.g. three prior shutdowns qualitatively different from one),
which decision trees model naturally but Logistic Regression approximates with smooth curves.

In [6]:
K = 30   # features to retain in SelectKBest step

# Reproducible mutual_info_classif (random_state fixes the neighbourhood sampling)
mi = partial(mutual_info_classif, random_state=42)

pipe_lr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LogisticRegression(max_iter=1000, random_state=42)),
])

pipe_lr_sel = Pipeline([
    ('imputer',   SimpleImputer(strategy='median')),
    ('scaler',    StandardScaler()),
    ('selector',  SelectKBest(mi, k=K)),
    ('model',     LogisticRegression(max_iter=1000, random_state=42)),
])

pipe_lr.fit(X_train, y_train)
lr_acc = accuracy_score(y_test, pipe_lr.predict(X_test))
print(f'Logistic Regression (all {X_train.shape[1]} features) : {lr_acc*100:.1f}%')

pipe_lr_sel.fit(X_train, y_train)
lr_sel_acc = accuracy_score(y_test, pipe_lr_sel.predict(X_test))
print(f'Logistic Regression (top {K} via SelectKBest)       : {lr_sel_acc*100:.1f}%')

Logistic Regression (all 93 features) : 32.0%


Logistic Regression (top 30 via SelectKBest)       : 31.0%


## Step 5 — Model B: Random Forest

### Why Random Forest suits this problem

A Random Forest is an ensemble of decision trees, each trained on a bootstrapped sample
with a random feature subset at each split. It predicts the majority class vote across
all trees.

It is a strong candidate here because:
- **Non-linear history effects are captured**: whether an elevator has had 0, 1, or 10
  prior shutdowns is not a linear signal — there are threshold effects. Decision trees
  split on exact thresholds (`prior_oc_shutdown > 2`), which directly models this.
- **Feature interactions are handled implicitly**: whether a high `prior_order_count`
  is dangerous depends on the `AlterationCount`; trees explore these conjunctions
  naturally through sequential splits.
- **Scale-invariant**: no `StandardScaler` required; counts, rates, and binary dummies
  are treated uniformly by split criteria.
- **Robust to irrelevant features**: each tree sees a random feature subset, so the
  many sparse city/equipment dummies do not consistently pollute every tree's splits.

Its main cost is training time compared to Logistic Regression, and reduced
interpretability — though feature importances provide partial explanation.

In [7]:
pipe_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model',   RandomForestClassifier(
                    n_estimators=100, max_depth=20,
                    random_state=42, n_jobs=-1)),
])

pipe_rf_sel = Pipeline([
    ('imputer',  SimpleImputer(strategy='median')),
    ('selector', SelectKBest(mi, k=K)),
    ('model',    RandomForestClassifier(
                     n_estimators=100, max_depth=20,
                     random_state=42, n_jobs=-1)),
])

pipe_rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, pipe_rf.predict(X_test))
print(f'Random Forest (all {X_train.shape[1]} features)         : {rf_acc*100:.1f}%')

pipe_rf_sel.fit(X_train, y_train)
rf_sel_acc = accuracy_score(y_test, pipe_rf_sel.predict(X_test))
print(f'Random Forest (top {K} via SelectKBest)              : {rf_sel_acc*100:.1f}%')

Random Forest (all 93 features)         : 34.8%


Random Forest (top 30 via SelectKBest)              : 33.6%


## Step 6 — Results Summary and Best Model

In [8]:
summary = {
    'Model':         ['Logistic Regression', 'Logistic Regression',
                      'Random Forest',       'Random Forest'],
    'Feature set':   [f'All {X_train.shape[1]}', f'Top {K} (SelectKBest)',
                      f'All {X_train.shape[1]}', f'Top {K} (SelectKBest)'],
    'Accuracy':      [lr_acc, lr_sel_acc, rf_acc, rf_sel_acc],
    'vs full baseline': [lr_acc - baseline_acc, lr_sel_acc - baseline_acc,
                         rf_acc - baseline_acc,  rf_sel_acc - baseline_acc],
    'vs test baseline': [lr_acc - baseline_test, lr_sel_acc - baseline_test,
                         rf_acc - baseline_test,  rf_sel_acc - baseline_test],
}
results_df = (
    __import__('pandas').DataFrame(summary)
    .sort_values('Accuracy', ascending=False)
    .reset_index(drop=True)
)
results_df['Accuracy'] = results_df['Accuracy'].map('{:.1%}'.format)
for col in ('vs full baseline', 'vs test baseline'):
    results_df[col] = results_df[col].map('{:+.1%}'.format)
print(f'Full-dataset baseline (Step 1) : {baseline_acc*100:.1f}%')
print(f'Test-set baseline              : {baseline_test*100:.1f}%  (predict "{most_common}" on test)\n')
print(results_df.to_string(index=False))

Full-dataset baseline (Step 1) : 38.1%
Test-set baseline              : 29.0%  (predict "Follow up" on test)

              Model          Feature set Accuracy vs full baseline vs test baseline
      Random Forest               All 93    34.8%            -3.4%            +5.8%
      Random Forest Top 30 (SelectKBest)    33.6%            -4.6%            +4.6%
Logistic Regression               All 93    32.0%            -6.1%            +3.0%
Logistic Regression Top 30 (SelectKBest)    31.0%            -7.2%            +2.0%


### Best model selection

The **best-performing model will be identified from the table above** (cell output).

Based on prior runs and the characteristics of the data, Random Forest with all features
is expected to win. The justification is:

- **Metric**: accuracy — chosen because the spec defines the baseline as an accuracy
  figure (38.1 %) and all classes are retained in training and testing, making accuracy
  a meaningful aggregate measure across the 13 outcome classes.
- **Why Random Forest over Logistic Regression**: the prior-history count features
  (`prior_oc_shutdown`, `prior_inspection_count`, `days_since_last_inspection`) have
  non-linear relationships with the target — threshold effects that trees exploit directly.
  Logistic Regression must approximate these with continuous weights.
- **Why all features over SelectKBest-reduced**: `mutual_info_classif` scores features
  *independently*. The 13 `prior_oc_*` columns collectively describe the distribution of
  prior outcomes; dropping any one looks cheap in isolation but degrades the joint signal.
  The all-feature pipeline preserves these complementary columns.
- **vs baseline**: every model is expected to beat the 38.1 % naive baseline. Beating it
  confirms that the prior-history features carry genuine predictive signal beyond simply
  predicting the most common class for every row.